# Phase 3 - Exploratory Data Analysis (EDA)

**Input:** the clean Parquet tables from Phase 2 (`MyDrive/amazon_reviews/parquet/*_1pct`)
**Output:**
- report-ready charts in `MyDrive/amazon_reviews/results/figures/`
- the numbers behind every chart as CSV in `MyDrive/amazon_reviews/results/eda/` (the dashboard will reuse them)

| # | Question | Mainly useful for |
|---|---|---|
| 1 | Overview numbers | report, all |
| 2 | How are star ratings spread? | sentiment (Harshad) |
| 3 | How did reviews and ratings change over the years? | report |
| 4 | Which months get the most reviews? | report |
| 5 | Do happy and unhappy customers write different lengths? | sentiment |
| 6 | Do verified and unverified buyers rate differently? | fake reviews (Kanal) |
| 7 | Which product categories get the most reviews? | report |
| 8 | How many reviews are voted helpful? | report |
| 9 | How many reviews does a typical user / product have? | recommendation (Prathamesh) |
| 10 | How much data survives 5-core filtering? | recommendation |
| 11 | Early look at fake-review signals | fake reviews |
| 12 | Most common words in positive vs negative reviews | sentiment |

All the heavy work is done by **Spark** (Spark SQL and DataFrames). Only the small result
tables are turned into pandas for drawing charts - that is how EDA is done on big data.

Run the cells **top to bottom** (about 10-15 minutes, mostly setup).

## 0. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT = '/content/drive/MyDrive/amazon_reviews'
SAMPLE = '1pct'
LOCAL = '/content/parquet'
FIG_DIR = f'{PROJECT}/results/figures'
EDA_DIR = f'{PROJECT}/results/eda'

In [ ]:
import os
!apt-get -qq update > /dev/null && apt-get -qq install -y openjdk-17-jdk-headless > /dev/null
os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-17-openjdk-amd64'
!pip -q install pyspark==3.5.3

In [ ]:
# copy the Parquet tables to the local disk (faster than reading from Drive)
!mkdir -p {LOCAL} {FIG_DIR} {EDA_DIR}
!cp -r {PROJECT}/parquet/reviews_clean_{SAMPLE} {PROJECT}/parquet/products_clean_{SAMPLE} {LOCAL}/

from pyspark.sql import SparkSession, functions as F

spark = (SparkSession.builder
         .appName('Phase3-EDA')
         .master('local[*]')
         .config('spark.driver.memory', '8g')
         .config('spark.sql.shuffle.partitions', '16')
         .getOrCreate())
spark.sparkContext.setLogLevel('ERROR')

reviews = spark.read.parquet(f'{LOCAL}/reviews_clean_{SAMPLE}').cache()
products = spark.read.parquet(f'{LOCAL}/products_clean_{SAMPLE}').cache()
reviews.createOrReplaceTempView('reviews')
products.createOrReplaceTempView('products')
print(f'reviews: {reviews.count():,}   products: {products.count():,}')

### Chart style

One consistent style for every chart in the report:
- **blue = positive, grey = neutral, red = negative** (same everywhere)
- light grid, no top/right borders, titles on the left
- every chart is saved as PNG (for the report) and its numbers as CSV (for the dashboard)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

BLUE, ORANGE, AQUA, RED, GREY = '#2a78d6', '#eb6834', '#1baf7a', '#e34948', '#9a9893'
TEXT, MUTED, GRID = '#0b0b0b', '#52514e', '#e6e5e0'
SENT_COLORS = {'positive': BLUE, 'neutral': GREY, 'negative': RED}
SENT_ORDER = ['positive', 'neutral', 'negative']
RATING_COLORS = {1: RED, 2: RED, 3: GREY, 4: BLUE, 5: BLUE}

plt.rcParams.update({
    'figure.dpi': 110, 'savefig.dpi': 200, 'font.size': 10,
    'axes.edgecolor': GRID, 'axes.labelcolor': MUTED,
    'xtick.color': MUTED, 'ytick.color': MUTED,
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'axes.grid.axis': 'y', 'axes.ymargin': 0.12, 'grid.color': GRID, 'grid.linewidth': 0.8, 'axes.axisbelow': True,
    'axes.titlesize': 12, 'axes.titleweight': 'bold', 'axes.titlecolor': TEXT, 'axes.titlelocation': 'left',
    'legend.frameon': False,
})

def save_fig(fig, name):
    fig.tight_layout()
    fig.savefig(f'{FIG_DIR}/{name}.png', bbox_inches='tight', facecolor='white')
    plt.show()

def save_table(pdf, name):
    pdf.to_csv(f'{EDA_DIR}/{name}.csv', index=False)

def thousands(ax, axis='y'):
    target = ax.yaxis if axis == 'y' else ax.xaxis
    target.set_major_locator(mtick.MaxNLocator(5))   # at most ~5 ticks, so labels never collide
    target.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'{x:,.0f}'))

## 1. Overview numbers

In [ ]:
overview = spark.sql('''
    SELECT COUNT(*)                                          AS reviews,
           COUNT(DISTINCT user_id)                           AS users,
           COUNT(DISTINCT parent_asin)                       AS products,
           MIN(review_date)                                  AS first_review,
           MAX(review_date)                                  AS last_review,
           ROUND(AVG(rating), 2)                             AS avg_rating,
           ROUND(100 * AVG(CAST(verified_purchase AS INT)), 1) AS verified_pct,
           ROUND(AVG(word_count), 1)                         AS avg_words,
           ROUND(COUNT(*) / COUNT(DISTINCT user_id), 2)      AS reviews_per_user,
           ROUND(COUNT(*) / COUNT(DISTINCT parent_asin), 2)  AS reviews_per_product
    FROM reviews
''').toPandas()
save_table(overview, '01_overview')
overview.T.rename(columns={0: 'value'})

## 2. Star rating distribution

In [ ]:
ratings = spark.sql('''
    SELECT CAST(rating AS INT) AS rating, COUNT(*) AS reviews
    FROM reviews GROUP BY 1 ORDER BY 1
''').toPandas()
ratings['pct'] = (100 * ratings['reviews'] / ratings['reviews'].sum()).round(1)
save_table(ratings, '02_rating_distribution')
ratings

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(ratings['rating'].astype(str) + ' star', ratings['pct'], width=0.6,
              color=[RATING_COLORS[r] for r in ratings['rating']], edgecolor='white', linewidth=2)
ax.bar_label(bars, labels=[f'{p:.1f}%' for p in ratings['pct']], padding=3, color=TEXT)
ax.yaxis.set_major_formatter(mtick.PercentFormatter(decimals=0))
ax.set_ylabel('share of reviews')
ax.set_title('Most reviews are 5-star: the classes are imbalanced')
save_fig(fig, '02_rating_distribution')

## 3. Reviews and average rating per year

Two separate panels (not one chart with two y-axes) so each number has its own scale.
Average rating is only shown for years with at least 100 reviews - with fewer it jumps around by chance.
The dataset ends in **September 2023**, so 2023 is not a full year.

In [ ]:
yearly = spark.sql('''
    SELECT year, COUNT(*) AS reviews, ROUND(AVG(rating), 3) AS avg_rating,
           ROUND(100 * AVG(CASE WHEN sentiment = 'negative' THEN 1 ELSE 0 END), 1) AS negative_pct
    FROM reviews GROUP BY year ORDER BY year
''').toPandas()
save_table(yearly, '03_reviews_per_year')
yearly

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(9, 6), sharex=True, gridspec_kw={'height_ratios': [3, 2]})

ax1.bar(yearly['year'], yearly['reviews'], width=0.7, color=BLUE, edgecolor='white', linewidth=1)
thousands(ax1)
ax1.set_ylabel('reviews')
ax1.set_title('Reviews per year (2023 = Jan to Sep only)')
peak = yearly.loc[yearly['reviews'].idxmax()]
ax1.annotate(f"peak {int(peak['year'])}: {int(peak['reviews']):,}", (peak['year'], peak['reviews']),
             xytext=(0, 4), textcoords='offset points', ha='center', color=TEXT)

big = yearly[yearly['reviews'] >= 100]
ax2.plot(big['year'], big['avg_rating'], color=BLUE, linewidth=2, marker='o', markersize=5)
ax2.set_ylabel('average stars')
ax2.set_title('Average rating per year (years with 100+ reviews)')
ax2.xaxis.set_major_locator(mtick.MaxNLocator(integer=True))
save_fig(fig, '03_reviews_per_year')

## 4. Reviews by month of the year (seasonality)

In [ ]:
monthly = spark.sql('''
    SELECT MONTH(review_date) AS month, COUNT(*) AS reviews
    FROM reviews GROUP BY 1 ORDER BY 1
''').toPandas()
monthly['pct'] = (100 * monthly['reviews'] / monthly['reviews'].sum()).round(2)
save_table(monthly, '04_reviews_by_month')
monthly

In [ ]:
month_names = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
fig, ax = plt.subplots(figsize=(9, 3.8))
bars = ax.bar([month_names[m - 1] for m in monthly['month']], monthly['pct'], width=0.6,
              color=BLUE, edgecolor='white', linewidth=2)
top = monthly['pct'].idxmax()
ax.bar_label(bars, labels=[f'{p:.1f}%' if i == top else '' for i, p in enumerate(monthly['pct'])],
             padding=3, color=TEXT)
ax.axhline(100 / 12, color=MUTED, linewidth=1, linestyle='--')
ax.text(len(monthly) - 0.5, 100 / 12, ' even spread', va='bottom', ha='right', color=MUTED, fontsize=9)
ax.yaxis.set_major_formatter(mtick.PercentFormatter(decimals=0))
ax.set_ylabel('share of reviews')
ax.set_title('Share of reviews by month')
save_fig(fig, '04_reviews_by_month')

## 5. Review length by sentiment

`word_count` is the number of words **after** stop words were removed (from Phase 2).

In [ ]:
length_stats = spark.sql('''
    SELECT sentiment, COUNT(*) AS reviews,
           ROUND(AVG(word_count), 1)                      AS mean_words,
           PERCENTILE_APPROX(word_count, 0.5)            AS median_words,
           PERCENTILE_APPROX(word_count, 0.9)            AS p90_words
    FROM reviews GROUP BY sentiment
''').toPandas().set_index('sentiment').loc[SENT_ORDER].reset_index()
save_table(length_stats, '05_length_stats')

length_buckets = spark.sql('''
    SELECT sentiment,
           CASE WHEN word_count <= 5   THEN '1-5'
                WHEN word_count <= 15  THEN '6-15'
                WHEN word_count <= 30  THEN '16-30'
                WHEN word_count <= 60  THEN '31-60'
                WHEN word_count <= 120 THEN '61-120'
                ELSE '120+' END AS words,
           COUNT(*) AS reviews
    FROM reviews GROUP BY 1, 2
''').toPandas()
length_buckets['pct'] = 100 * length_buckets['reviews'] / length_buckets.groupby('sentiment')['reviews'].transform('sum')
save_table(length_buckets, '05_length_buckets')
length_stats

In [ ]:
BUCKETS = ['1-5', '6-15', '16-30', '31-60', '61-120', '120+']
pivot = length_buckets.pivot(index='words', columns='sentiment', values='pct').reindex(BUCKETS).fillna(0)

fig, ax = plt.subplots(figsize=(9, 4))
width = 0.26
for i, s in enumerate(SENT_ORDER):
    ax.bar([x + (i - 1) * width for x in range(len(BUCKETS))], pivot[s], width=width,
           color=SENT_COLORS[s], edgecolor='white', linewidth=1.5, label=s)
ax.set_xticks(range(len(BUCKETS)), BUCKETS)
ax.set_xlabel('words in review (after removing stop words)')
ax.yaxis.set_major_formatter(mtick.PercentFormatter(decimals=0))
ax.set_ylabel('share of reviews in that sentiment')
ax.set_title('Negative and neutral reviews are longer than positive ones')
ax.legend(ncols=3, loc='upper right')
save_fig(fig, '05_review_length')

## 6. Verified vs unverified purchases

A **verified purchase** means Amazon confirmed the reviewer bought the product.
Unverified reviews are one of the fake-review signals in the synopsis - here we only check
whether they *look different*, not whether they are fake.

In [ ]:
verified = spark.sql('''
    SELECT CASE WHEN verified_purchase THEN 'verified' ELSE 'unverified' END AS purchase,
           CAST(rating AS INT) AS rating, COUNT(*) AS reviews
    FROM reviews GROUP BY 1, 2
''').toPandas()
verified['pct'] = 100 * verified['reviews'] / verified.groupby('purchase')['reviews'].transform('sum')
save_table(verified, '06_verified_ratings')

verified_summary = spark.sql('''
    SELECT CASE WHEN verified_purchase THEN 'verified' ELSE 'unverified' END AS purchase,
           COUNT(*) AS reviews,
           ROUND(100 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS share_pct,
           ROUND(AVG(rating), 2) AS avg_rating,
           ROUND(AVG(word_count), 1) AS avg_words,
           ROUND(AVG(helpful_vote), 2) AS avg_helpful_votes
    FROM reviews GROUP BY 1
''').toPandas()
save_table(verified_summary, '06_verified_summary')
verified_summary

In [ ]:
pivot = verified.pivot(index='rating', columns='purchase', values='pct').reindex([1, 2, 3, 4, 5]).fillna(0)
fig, ax = plt.subplots(figsize=(8, 4))
width = 0.38
for i, (group, color) in enumerate([('verified', BLUE), ('unverified', ORANGE)]):
    ax.bar([r + (i - 0.5) * width for r in pivot.index], pivot[group], width=width,
           color=color, edgecolor='white', linewidth=1.5, label=group)
ax.set_xticks([1, 2, 3, 4, 5], ['1 star', '2 star', '3 star', '4 star', '5 star'])
ax.yaxis.set_major_formatter(mtick.PercentFormatter(decimals=0))
ax.set_ylabel('share of reviews in that group')
ax.set_title('Star ratings: verified vs unverified buyers')
ax.legend(loc='upper left')
save_fig(fig, '06_verified_vs_unverified')

## 7. Top product categories

`main_category` comes from the product metadata (joined on `parent_asin`).

In [ ]:
categories = spark.sql('''
    SELECT p.main_category, COUNT(*) AS reviews,
           COUNT(DISTINCT r.parent_asin) AS products,
           ROUND(AVG(r.rating), 2) AS avg_rating
    FROM reviews r JOIN products p ON r.parent_asin = p.parent_asin
    GROUP BY p.main_category ORDER BY reviews DESC
''').toPandas()
save_table(categories, '07_categories')
categories.head(15)

In [ ]:
top = categories.head(12).iloc[::-1]
fig, ax = plt.subplots(figsize=(9, 5.5))
bars = ax.barh(top['main_category'], top['reviews'], height=0.6, color=BLUE, edgecolor='white', linewidth=1.5)
ax.bar_label(bars, labels=[f"  avg {r:.2f} stars" for r in top['avg_rating']], color=MUTED, fontsize=9)
thousands(ax, 'x')
ax.set_xlabel('reviews')
ax.set_xlim(0, top['reviews'].max() * 1.25)
ax.grid(axis='x', visible=True)
ax.grid(axis='y', visible=False)
ax.set_title('Top 12 categories by number of reviews')
save_fig(fig, '07_top_categories')

## 8. Helpful votes

In [ ]:
helpful = spark.sql('''
    SELECT sentiment,
           COUNT(*) AS reviews,
           ROUND(100 * AVG(CASE WHEN helpful_vote = 0 THEN 1 ELSE 0 END), 1) AS zero_votes_pct,
           ROUND(AVG(helpful_vote), 2) AS avg_votes,
           MAX(helpful_vote) AS max_votes
    FROM reviews GROUP BY sentiment
''').toPandas().set_index('sentiment').loc[SENT_ORDER].reset_index()
save_table(helpful, '08_helpful_by_sentiment')

most_helpful = spark.sql('''
    SELECT r.helpful_vote, CAST(r.rating AS INT) AS rating, r.title AS review_title,
           p.product_title, r.word_count
    FROM reviews r LEFT JOIN products p ON r.parent_asin = p.parent_asin
    ORDER BY r.helpful_vote DESC LIMIT 10
''').toPandas()
save_table(most_helpful, '08_most_helpful_reviews')
helpful

In [ ]:
pd.set_option('display.max_colwidth', 60)
most_helpful

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.5))
bars = ax.bar(helpful['sentiment'], helpful['avg_votes'], width=0.55,
              color=[SENT_COLORS[s] for s in helpful['sentiment']], edgecolor='white', linewidth=2)
ax.bar_label(bars, labels=[f'{v:.2f}' for v in helpful['avg_votes']], padding=3, color=TEXT)
ax.set_ylabel('average helpful votes per review')
ax.set_title('Which reviews do other shoppers find helpful?')
save_fig(fig, '08_helpful_votes')

## 9. How many reviews does each user and each product have?

This is the **long tail**: most users write only one review and most products get only a few.
It matters a lot for the recommender - ALS learns from users who reviewed several products.

In [ ]:
def bucket_counts(key):
    return spark.sql(f'''
        SELECT CASE WHEN n = 1 THEN '1' WHEN n = 2 THEN '2' WHEN n <= 4 THEN '3-4'
                    WHEN n <= 9 THEN '5-9' WHEN n <= 19 THEN '10-19' ELSE '20+' END AS reviews_each,
               COUNT(*) AS count
        FROM (SELECT {key}, COUNT(*) AS n FROM reviews GROUP BY {key})
        GROUP BY 1
    ''').toPandas()

per_user = bucket_counts('user_id')
per_product = bucket_counts('parent_asin')
for pdf in (per_user, per_product):
    pdf['pct'] = (100 * pdf['count'] / pdf['count'].sum()).round(1)
save_table(per_user, '09_reviews_per_user')
save_table(per_product, '09_reviews_per_product')
per_user.merge(per_product, on='reviews_each', suffixes=('_users', '_products'))

In [ ]:
ORDER = ['1', '2', '3-4', '5-9', '10-19', '20+']
fig, axes = plt.subplots(1, 2, figsize=(10, 3.8), sharey=True)
for ax, pdf, who in [(axes[0], per_user, 'users'), (axes[1], per_product, 'products')]:
    pdf = pdf.set_index('reviews_each').reindex(ORDER).fillna(0)
    bars = ax.bar(ORDER, pdf['pct'], width=0.6, color=BLUE, edgecolor='white', linewidth=2)
    ax.bar_label(bars, labels=[f'{p:.0f}%' for p in pdf['pct']], padding=3, color=TEXT, fontsize=9)
    ax.set_xlabel('number of reviews')
    ax.set_title(f'Reviews per {who[:-1]}')
axes[0].yaxis.set_major_formatter(mtick.PercentFormatter(decimals=0))
axes[0].set_ylabel('share')
save_fig(fig, '09_long_tail')

## 10. How much data survives k-core filtering? (for the recommender)

**k-core** = keep only users with at least *k* ratings **and** products with at least *k* ratings.
Removing products can push some users below *k*, so we repeat until nothing changes.

The synopsis plans **5-core** for ALS. This tells Prathamesh whether the 1% sample is big enough.

In [ ]:
# one rating per (user, product); a few users reviewed the same product twice
pairs = (reviews.groupBy('user_id', 'parent_asin')
                .agg(F.avg('rating').alias('rating'))
                .cache())
n_pairs = pairs.count()

def k_core(df, k, max_rounds=20):
    n_old = df.count()
    for _ in range(max_rounds):
        good_users = df.groupBy('user_id').count().filter(F.col('count') >= k).select('user_id')
        good_items = df.groupBy('parent_asin').count().filter(F.col('count') >= k).select('parent_asin')
        df = df.join(good_users, 'user_id').join(good_items, 'parent_asin').localCheckpoint()
        n_new = df.count()
        if n_new == n_old:
            break
        n_old = n_new
    return df

rows = []
for k in [1, 2, 3, 5, 10]:
    core = pairs if k == 1 else k_core(pairs, k)
    n = core.count()
    u = core.select('user_id').distinct().count()
    p = core.select('parent_asin').distinct().count()
    rows.append({'k': k, 'ratings': n, 'users': u, 'products': p,
                 'ratings_kept_pct': round(100 * n / n_pairs, 1),
                 'density_pct': round(100 * n / (u * p), 4) if u and p else 0})
    print(f'{k:>2}-core: {n:>9,} ratings  {u:>8,} users  {p:>8,} products')

kcore = pd.DataFrame(rows)
save_table(kcore, '10_kcore')
kcore

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.8))
labels = [f'{k}-core' if k > 1 else 'all' for k in kcore['k']]
colors = [ORANGE if k == 5 else BLUE for k in kcore['k']]
bars = ax.bar(labels, kcore['ratings'], width=0.6, color=colors, edgecolor='white', linewidth=2)
ax.bar_label(bars, labels=[f'{n:,}' for n in kcore['ratings']], padding=3, color=TEXT, fontsize=9)
thousands(ax)
ax.set_ylabel('user-product ratings left')
ax.set_title('Ratings left after k-core filtering (orange = 5-core used by ALS)')
save_fig(fig, '10_kcore')

**How to read this (for Prathamesh):** ALS needs enough overlap between users. As a rough guide,
if the 5-core row has **fewer than about 20,000 ratings**, the 1% sample is too thin for meaningful
recommendation results. Then use either a **5% sample** (`--percent 5` in Phase 1) or the
**official 5-core ratings file** from the dataset site for the recommendation module only.

## 11. Early look at fake-review signals (for Kanal)

These are **not** final fake-review results - just a first check that the signals from the
synopsis actually show up in the data.

| Signal | What we check here |
|---|---|
| Copied text | the same review text (5+ words) posted on **different** products |
| One-sided reviewers | users with 5+ reviews who give **only** 5 stars or **only** 1 star |
| Bursts | many reviews for one product on a **single day** |
| Rating deviation | reviews 3+ stars away from the product's average rating |

In [ ]:
copied = spark.sql('''
    SELECT text_clean,
           COUNT(*) AS reviews,
           COUNT(DISTINCT parent_asin) AS products,
           COUNT(DISTINCT user_id) AS users,
           ROUND(AVG(rating), 2) AS avg_rating
    FROM reviews
    WHERE word_count >= 5
    GROUP BY text_clean
    HAVING COUNT(DISTINCT parent_asin) >= 2
    ORDER BY reviews DESC
''').cache()

n_texts = copied.count()
n_reviews = copied.agg(F.sum('reviews')).first()[0] or 0
print(f'Texts posted on 2+ different products: {n_texts:,}  (covering {n_reviews:,} reviews)')
copied_top = copied.limit(15).toPandas()
save_table(copied_top, '11_copied_text_top')
copied_top

In [ ]:
one_sided = spark.sql('''
    SELECT COUNT(*) AS users_with_5plus_reviews,
           SUM(CASE WHEN min_r = 5 THEN 1 ELSE 0 END) AS only_5_star,
           SUM(CASE WHEN max_r = 1 THEN 1 ELSE 0 END) AS only_1_star
    FROM (SELECT user_id, MIN(rating) AS min_r, MAX(rating) AS max_r
          FROM reviews GROUP BY user_id HAVING COUNT(*) >= 5)
''').toPandas()
one_sided['only_5_star_pct'] = (100 * one_sided['only_5_star'] / one_sided['users_with_5plus_reviews']).round(1)
one_sided['only_1_star_pct'] = (100 * one_sided['only_1_star'] / one_sided['users_with_5plus_reviews']).round(1)
save_table(one_sided, '11_one_sided_reviewers')
one_sided

In [ ]:
bursts = spark.sql('''
    SELECT d.parent_asin, p.product_title, d.review_date, d.reviews_that_day, d.avg_rating
    FROM (SELECT parent_asin, review_date, COUNT(*) AS reviews_that_day, ROUND(AVG(rating), 2) AS avg_rating
          FROM reviews GROUP BY parent_asin, review_date) d
    LEFT JOIN products p ON d.parent_asin = p.parent_asin
    ORDER BY d.reviews_that_day DESC
    LIMIT 15
''').toPandas()
save_table(bursts, '11_top_bursts')
bursts

In [ ]:
deviation = spark.sql('''
    SELECT CASE WHEN ABS(r.rating - p.average_rating) < 1 THEN '0-1'
                WHEN ABS(r.rating - p.average_rating) < 2 THEN '1-2'
                WHEN ABS(r.rating - p.average_rating) < 3 THEN '2-3'
                ELSE '3+' END AS stars_from_product_avg,
           COUNT(*) AS reviews
    FROM reviews r JOIN products p ON r.parent_asin = p.parent_asin
    WHERE p.average_rating IS NOT NULL
    GROUP BY 1 ORDER BY 1
''').toPandas()
deviation['pct'] = (100 * deviation['reviews'] / deviation['reviews'].sum()).round(1)
save_table(deviation, '11_rating_deviation')
deviation

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.6))
colors = [RED if b == '3+' else BLUE for b in deviation['stars_from_product_avg']]
bars = ax.bar(deviation['stars_from_product_avg'], deviation['pct'], width=0.55,
              color=colors, edgecolor='white', linewidth=2)
ax.bar_label(bars, labels=[f'{p:.1f}%' for p in deviation['pct']], padding=3, color=TEXT)
ax.yaxis.set_major_formatter(mtick.PercentFormatter(decimals=0))
ax.set_xlabel("stars away from the product's average rating")
ax.set_ylabel('share of reviews')
ax.set_title('Rating deviation (red = 3+ stars away, a suspicion signal)')
save_fig(fig, '11_rating_deviation')

## 12. Most common words: positive vs negative

`explode` turns each list of words into one row per word, then Spark counts them.
Words shorter than 3 letters are skipped.

In [ ]:
words = (reviews
         .filter(F.col('sentiment').isin('positive', 'negative'))
         .select('sentiment', F.explode('tokens').alias('word'))
         .filter(F.length('word') >= 3)
         .groupBy('sentiment', 'word').count())

top_words = {}
for s in ['positive', 'negative']:
    top_words[s] = words.filter(F.col('sentiment') == s).orderBy(F.desc('count')).limit(20).toPandas()
    save_table(top_words[s], f'12_top_words_{s}')
pd.concat({s: top_words[s][['word', 'count']] for s in top_words}, axis=1)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 5.5))
for ax, s in zip(axes, ['positive', 'negative']):
    pdf = top_words[s].head(15).iloc[::-1]
    ax.barh(pdf['word'], pdf['count'], height=0.6, color=SENT_COLORS[s], edgecolor='white', linewidth=1.5)
    thousands(ax, 'x')
    ax.grid(axis='x', visible=True)
    ax.grid(axis='y', visible=False)
    ax.set_title(f'Top words in {s} reviews')
save_fig(fig, '12_top_words')

## Files created

In [ ]:
print('Figures:'); print('\n'.join(sorted(os.listdir(FIG_DIR))))
print('\nTables:'); print('\n'.join(sorted(os.listdir(EDA_DIR))))

## Done - send back these outputs

1. Section 1 (overview)
2. Section 6 table (verified summary)
3. Section 10 table (k-core) - decides the recommender data
4. Section 11 outputs (copied text count, one-sided reviewers, bursts, deviation)

**Next:** Phase 4 - the three modules (sentiment, fake reviews, recommendation) in parallel.